### 17. Letter Combinations of a Phone Number (Phone Number Translation)

#### Question
```
Write a function that takes a 7-digit telephone number and prints out all the
possible "words" or combinations of letters that can represent the given
numbers, according to the letters associated with each key on the telephone
keypad.
```

```
   1        2        3   
          (ABC)    (DEF) 
   4        5        6   
 (GHI)    (JKL)    (MNO) 
   7        8        9   
(PQRS)    (TUV)    (WXYZ)
   #        0        *   
           (_)           
```

#### Clarifying Questions
- What characters are allowed in the input?
  - Input string will contain digits only: no letters or punctuation

#### Example
Example phone_number and resulting letter combinations, using only 3 digits here for brevity, real inputs have 7 digits.

| phone_number | result |
| :--- | :--- |
| 42 | ga, gb, gc, ha, hb, hc, ia, ib, ic |
| 258 | ajt, bjt, cjt, akt, bkt, ckt, alt, blt, clt, aju, bju, cju, aku, bku, cku, alu, blu, clu, ajv, bjv, cjv, akv, bkv, ckv, alv, blv, clv |

#### Follow-Up Questions / Additional Discussion
```
Print only those solutions that are words or combination of words
present in a dictionary.
```

- 時間複雜度：$O(4^N \cdot N)$
  - 雖然加入了跳過邏輯，但在最壞情況下（數字全是 7 或 9），每個數字對應4個字母，最多產生 $4^N$ 個組合。
  - 每個組合最後用 "".join() 結合成字串需要 $O(N)$。
- 空間複雜度：$O(N)$
  - 遞迴深度最大為輸入字串長度 $N$，系統棧空間與 paths 列表皆為 $O(N)$。

In [1]:
class Solution:
    def letterCombinations(self, digits):
        # edge case: 如果輸入字串為空，回傳空列表
        if not digits:
            return []
        
        # 建立數字對應字母的字典
        phone_map = {
            "2": "abc", 
            "3": "def", 
            "4": "ghi", 
            "5": "jkl", 
            "6": "mno", 
            "7": "pqrs", 
            "8": "tuv", 
            "9": "wxyz"
        }

        result = []

        def dfs(index, paths):
            # 終止條件：處理完所有輸入字元，直接加入結果
            if index == len(digits):
                word = "".join(paths)
                result.append(word)
                return
            
            digit = digits[index]
            # 使用 get 取得對應字母，若無效（如 '1' or '#'）則回傳空字串
            letters = phone_map.get(digit, "")
            
            if letters:
                # 遍歷有效按鍵的所有字母
                for char in letters:
                    paths.append(char) # 新增該輪字母
                    dfs(index + 1, paths) # 遞迴
                    paths.pop() # 回溯
            else:
                # 跳過無效字元（Skip 邏輯），繼續處理下一個 index
                dfs(index + 1, paths)

        # 啟動深度優先搜尋
        dfs(index=0, paths=[])

        # edge case: 過濾掉空字串（針對輸入全是無效字元的情況, ex: "1_*"）
        result = [word for word in result if word]

        return result

In [2]:
digits = "2518"
Solution().letterCombinations(digits)

['ajt',
 'aju',
 'ajv',
 'akt',
 'aku',
 'akv',
 'alt',
 'alu',
 'alv',
 'bjt',
 'bju',
 'bjv',
 'bkt',
 'bku',
 'bkv',
 'blt',
 'blu',
 'blv',
 'cjt',
 'cju',
 'cjv',
 'ckt',
 'cku',
 'ckv',
 'clt',
 'clu',
 'clv']

如果產生了幾萬個組合，但字典裡只有 10 個字是真的，傳統寫法會浪費很多資源。這時可以使用 Trie 結合回溯 來處理「大數據過濾」。

- 時間複雜度：建立 Trie：$O(W \cdot L + 4^N \cdot N)$
  - 其中 $W$ 是字典單字數量。
  - $L$ 是字典單字平均長度。
  - 每個組合最後用 "".join() 結合成字串需要 $O(N)$。
  - 搜尋：最壞情況仍是 $O(4^N \cdot N)$，但平均情況下大幅優於基礎版，因為大量的無效組合被 Trie 剪枝掉，實際執行次數會大幅減少。
- 空間複雜度：$O(W \cdot L + N)$ 
  - 用於儲存 Trie 樹。
  - 遞迴深度仍為 $O(N)$。

In [3]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_word = False

class Solution:
    def letterCombinationsWithDict(self, digits: str, dictionary: list[str]) -> list[str]:
        if not digits: 
            return []

        # 1. 預處理：將字典建立成 Trie
        root = TrieNode()
        for word in dictionary:
            node = root
            for char in word:
                if char not in node.children:
                    node.children[char] = TrieNode()
                node = node.children[char]
            node.is_word = True

        phone_map = {
            "2": "abc", "3": "def", "4": "ghi", "5": "jkl",
            "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"
        }
        
        result = []

        # 2. 深度優先搜尋 (DFS) 結合 Trie 節點
        def dfs(index, path, curr_node):
            # 如果目前的前綴已經不在 Trie 中，直接剪枝返回
            if not curr_node:
                return

            # 終止條件：處理完所有數字
            if index == len(digits):
                # 只有當該路徑在字典中標記為完整單字時，才加入結果
                if curr_node.is_word:
                    result.append("".join(path))
                return

            digit = digits[index]
            letters = phone_map.get(digit, "")

            if letters:
                for char in letters:
                    # 只有當 Trie 的當前節點有這個字母的子節點時，才繼續往下走
                    if char in curr_node.children:
                        path.append(char)
                        dfs(index + 1, path, curr_node.children[char])
                        path.pop() # 回溯
            else:
                # 跳過無效字元，Trie 節點保持不變往下傳
                dfs(index + 1, path, curr_node)

        dfs(0, [], root)
        
        return result

In [4]:
dictionary = ["apple", "tree", "alt", "ball", "ckv"]
digits = "2518"
Solution().letterCombinationsWithDict(digits, dictionary)

['alt', 'ckv']